# PlayerPulse — Databricks Delta Lake Extension

This notebook extends the PlayerPulse pipeline with a second, parallel ingestion
path built on **Databricks**, **Delta Lake**, and **Unity Catalog**, applying the
same Bronze/Silver/Gold (medallion) pattern that the existing Snowflake/dbt
pipeline implements as RAW/STAGING/MARTS. It reads from the same raw S3 bucket
through a Unity Catalog External Location, and each layer is registered as a
managed Delta table in Unity Catalog.

In [0]:
raw_path = "s3://playerpulse-ali-jazz-raw-2026-218484443553-ca-central-1-an/chesscom/games/"

## Bronze — Raw ingestion

Reads the raw chess.com JSON archives directly from the S3 External Location
(`playerpulse_s3_raw`) using Spark's JSON reader with `multiLine`, since each
file is a single pretty-printed JSON object rather than newline-delimited JSON.
Partition discovery on the `username=/year=/month=` folder structure adds those
as columns automatically. An `ingested_at` timestamp is added, and the result
is written as a managed Delta table (`bronze.games_raw`) — one row per source
file, with the nested `games` array left untouched. This mirrors the role of
the RAW layer in the Snowflake pipeline.

In [0]:
df_bronze = spark.read.option("multiLine", "true").format("json").load(raw_path)

In [0]:
display(df_bronze)

games username year month List(List(List(85.2, 71.77), List(https://api.chess.com/pub/player/srinathkoneru, 677, win, srinathkoneru, 54d41166-13ab-11ef-87f5-b7bb74de3832), https://www.chess.com/openings/Vienna-Game, 1772611567, r4rk1/1b1pq1pp/n1p5/1p6/8/2P3Q1/P2K1PPP/8 w - - 0 26, rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1, [Event "Live Chess"]
[Site "Chess.com"]
[Date "2026.03.04"]
[Round "-"]
[White "ajaza"]
[Black "srinathkoneru"]
[Result "0-1"]
[CurrentPosition "r4rk1/1b1pq1pp/n1p5/1p6/8/2P3Q1/P2K1PPP/8 w - - 0 26"]
[Timezone "UTC"]
[ECO "C25"]
[ECOUrl "https://www.chess.com/openings/Vienna-Game"]
[UTCDate "2026.03.04"]
[UTCTime "07:55:36"]
[WhiteElo "538"]
[BlackElo "677"]
[TimeControl "600"]
[Termination "srinathkoneru won by resignation"]
[StartTime "07:55:36"]
[EndDate "2026.03.04"]
[EndTime "08:06:07"]
[Link "https://www.chess.com/game/live/165488023768"]

1. e4 {[%clk 0:09:57.4]} 1... e5 {[%clk 0:09:58.4]} 2. Nc3 {[%clk 0:09:53.1]} 2... Qf6 {[%clk 0:09:55.2]} 3. Bb5 {[%clk 0:09:19.1]} 3... Bc5 {[%clk 0:09:49.9]} 4. Nf3 {[%clk 0:09:02.3]} 4... c6 {[%clk 0:09:44.7]} 5. Ba4 {[%clk 0:08:37.4]} 5... b5 {[%clk 0:09:41.6]} 6. Bb3 {[%clk 0:08:30.5]} 6... a5 {[%clk 0:09:36.3]} 7. d3 {[%clk 0:08:08.4]} 7... a4 {[%clk 0:09:24.9]} 8. Bg5 {[%clk 0:08:02.3]} 8... Qg6 {[%clk 0:09:15.8]} 9. Bxf7+ {[%clk 0:06:46]} 9... Qxf7 {[%clk 0:09:05.3]} 10. d4 {[%clk 0:06:38.9]} 10... exd4 {[%clk 0:08:59.1]} 11. Nxd4 {[%clk 0:06:34.6]} 11... Bb4 {[%clk 0:08:42.6]} 12. Qg4 {[%clk 0:06:11.6]} 12... Na6 {[%clk 0:08:26.8]} 13. Bh4 {[%clk 0:05:55.4]} 13... a3 {[%clk 0:08:16.4]} 14. Qg5 {[%clk 0:05:37.2]} 14... Nf6 {[%clk 0:08:09.3]} 15. e5 {[%clk 0:05:08.8]} 15... axb2 {[%clk 0:08:00.6]} 16. Rb1 {[%clk 0:05:06.8]} 16... Bxc3+ {[%clk 0:07:57]} 17. Kd1 {[%clk 0:04:56.7]} 17... Bxd4 {[%clk 0:07:31.9]} 18. exf6 {[%clk 0:04:47.2]} 18... Bxf6 {[%clk 0:07:26.6]} 19. Qg3 {[%clk 0:04:40.5]} 19... O-O {[%clk 0:07:12.7]} 20. Bxf6 {[%clk 0:04:25.1]} 20... Qxf6 {[%clk 0:07:01.1]} 21. Re1 {[%clk 0:04:14.7]} 21... Bb7 {[%clk 0:06:57.6]} 22. Rxb2 {[%clk 0:04:01.2]} 22... Qxb2 {[%clk 0:06:51.9]} 23. Re7 {[%clk 0:03:59]} 23... Qb1+ {[%clk 0:06:30.1]} 24. Kd2 {[%clk 0:03:57.6]} 24... Qb4+ {[%clk 0:06:21.6]} 25. c3 {[%clk 0:03:49.1]} 25... Qxe7 {[%clk 0:06:18.9]} 0-1
, true, chess, null, mC0Kbs7TfH9IgvYQHyXHyrWGltGycMTUr1U1tBKBvBIzdE5OMFyqEM!TCKqjabzsedsBKTBTMw8!FT1The6XbjTje0jbdlbzksz0, rapid, 600, https://www.chess.com/game/live/165488023768, 878efddd-179f-11f1-9d7f-c1fc1601000f, List(https://api.chess.com/pub/player/ajaza, 538, resigned, ajaza, 728636c8-f032-11e8-801c-000000000000)), List(null, List(https://api.chess.com/pub/player/lass-schachen, 516, checkmated, lass-schachen, f55900aa-123d-11f1-8498-4db02845d7d1), https://www.chess.com/openings/Kings-Pawn-Opening-1...e5, 1772612168, r2k1b1r/pp1Q4/2p1P2p/8/4P1P1/2n2K2/2PR1PP1/2q3NR b - - 0 24, rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1, [Event "Live Chess"]
[Site "Chess.com"]
[Date "2026.03.04"]
[Round "-"]
[White "ajaza"]
[Black "lass-schachen"]
[Result "1-0"]
[CurrentPosition "r2k1b1r/pp1Q4/2p1P2p/8/4P1P1/2n2K2/2PR1PP1/2q3NR b - - 0 24"]
[Timezone "UTC"]
[ECO "C20"]
[ECOUrl "https://www.chess.com/openings/Kings-Pawn-Opening-1...e5"]
[UTCDate "2026.03.04"]
[UTCTime "08:07:13"]
[WhiteElo "623"]
[BlackElo "516"]
[TimeControl "600"]
[Termination "ajaza won by checkmate"]
[StartTime "08:07:13"]
[EndDate "2026.03.04"]
[EndTime "08:16:08"]
[Link "https://www.chess.com/game/live/165488325814"]

1. e4 {[%clk 0:09:55.8]} 1... e5 {[%clk 0:09:58.2]} 2. Qf3 {[%clk 0:09:47.3]} 2... Nf6 {[%clk 0:09:51.6]} 3. Nc3 {[%clk 0:09:42]} 3... c6 {[%clk 0:09:43.5]} 4. d3 {[%clk 0:09:26.2]} 4... d5 {[%clk 0:09:40.8]} 5. Bg5 {[%clk 0:09:21.5]} 5... Bg4 {[%clk 0:09:37.7]} 6. Qe3 {[%clk 0:08:54.8]} 6... d4 {[%clk 0:09:34.7]} 7. Qg3 {[%clk 0:08:48.2]} 7... dxc3 {[%clk 0:09:33.3]} 8. bxc3 {[%clk 0:08:44.3]} 8... h6 {[%clk 0:09:29.2]} 9. Bh4 {[%clk 0:08:33.8]} 9... g5 {[%clk 0:09:21.9]} 10. h3 {[%clk 0:08:15.9]} 10..

In [0]:
df_bronze.printSchema()

root
 |-- games: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- accuracies: struct (nullable = true)
 |    |    |    |-- black: double (nullable = true)
 |    |    |    |-- white: double (nullable = true)
 |    |    |-- black: struct (nullable = true)
 |    |    |    |-- @id: string (nullable = true)
 |    |    |    |-- rating: long (nullable = true)
 |    |    |    |-- result: string (nullable = true)
 |    |    |    |-- username: string (nullable = true)
 |    |    |    |-- uuid: string (nullable = true)
 |    |    |-- eco: string (nullable = true)
 |    |    |-- end_time: long (nullable = true)
 |    |    |-- fen: string (nullable = true)
 |    |    |-- initial_setup: string (nullable = true)
 |    |    |-- pgn: string (nullable = true)
 |    |    |-- rated: boolean (nullable = true)
 |    |    |-- rules: string (nullable = true)
 |    |    |-- start_time: long (nullable = true)
 |    |    |-- tcn: string (nullable = true)
 |    |    |-- time_

In [0]:
from pyspark.sql.functions import current_timestamp

df_bronze = df_bronze.withColumn("ingested_at", current_timestamp())

In [0]:
df_bronze.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.games_raw")

In [0]:
spark.read.table("workspace.bronze.games_raw").createOrReplaceTempView("bronze_games")

## Silver — Explode and type

Exposes the Bronze Delta table as a temp view, then explodes the nested `games`
array with `LATERAL VIEW EXPLODE` — the Spark SQL equivalent of Snowflake's
`LATERAL FLATTEN` — producing one row per game. The nested `white`/`black`/
`accuracies` structs are flattened into typed, named columns, directly mirroring
the dbt `stg_games` staging model field by field. Written as a managed Delta
table (`silver.games`).

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.games AS
SELECT
    game.uuid AS game_id,
    game.url AS game_url,
    timestamp_seconds(game.end_time) AS end_time_utc,
    game.rated AS is_rated,
    game.time_control AS time_control,
    game.rules AS rules,
    game.time_class AS time_class,
    game.eco AS opening_url,

    game.white.username AS white_username,
    game.white.rating AS white_rating,
    game.white.result AS white_result,
    game.accuracies.white AS white_accuracy,

    game.black.username AS black_username,
    game.black.rating AS black_rating,
    game.black.result AS black_result,
    game.accuracies.black AS black_accuracy,
    
    username AS source_username,
    year AS source_year,
    month AS source_month,
    ingested_at

FROM bronze_games
LATERAL VIEW EXPLODE(games) exploded_table AS game

num_affected_rows,num_inserted_rows


## Gold — Player perspective

Reframes the symmetric white/black columns into a player/opponent perspective
from Ali's point of view, filtered to games where `ajaza` participated. Derives
`game_outcome` (win/loss/draw) and `rating_difference`. This is a direct Spark
SQL port of the dbt `fct_player_games` mart — same logic, same output shape,
different engine. Written as a managed Delta table (`gold.fct_player_games`).

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.fct_player_games AS
with player_perspective as (
    SELECT
        game_id,
        game_url,
        end_time_utc,
        CAST(end_time_utc AS DATE) AS game_date,
        time_class,
        time_control,
        opening_url,
        is_rated,
        case
            when lower(white_username) = 'ajaza' then 'white'
            else 'black'
        end as player_color,

        case
            when lower(white_username) = 'ajaza' then white_username
            else black_username
        end as player_username,

        case
            when lower(white_username) = 'ajaza' then black_username
            else white_username
        end as opponent_username,

        case
            when lower(white_username) = 'ajaza' then white_rating
            else black_rating
        end as player_rating,

        case
            when lower(white_username) = 'ajaza' then black_rating
            else white_rating
        end as opponent_rating,

        case
            when lower(white_username) = 'ajaza' then white_result
            else black_result
        end as player_result,

        case
            when lower(white_username) = 'ajaza' then black_result
            else white_result
        end as opponent_result,

        case
            when lower(white_username) = 'ajaza' then white_accuracy
            else black_accuracy
        end as player_accuracy,

        case
            when lower(white_username) = 'ajaza' then black_accuracy
            else white_accuracy
        end as opponent_accuracy,

        source_username,
        source_year,
        source_month,
        ingested_at

FROM workspace.silver.games
WHERE lower(white_username) = 'ajaza' OR lower(black_username) = 'ajaza'
    ),

final as (

    select
        *,

        case
            when player_result = 'win' then 'win'
            when opponent_result = 'win' then 'loss'
            else 'draw'
        end as game_outcome,

        player_rating - opponent_rating as rating_difference

    from player_perspective

)

select *
from final

num_affected_rows,num_inserted_rows


## Data quality checks

Mirrors the intent of the project's dbt tests (not_null, unique, accepted_values)
as direct SQL assertions against the Gold table, plus one custom check comparing
the filtered Silver row count against the Gold row count, to confirm the
player-perspective transformation neither dropped nor duplicated rows.

In [0]:
%sql
SELECT *
FROM workspace.gold.fct_player_games
WHERE game_id IS NULL

game_id,game_url,end_time_utc,game_date,time_class,time_control,opening_url,is_rated,player_color,player_username,opponent_username,player_rating,opponent_rating,player_result,opponent_result,player_accuracy,opponent_accuracy,source_username,source_year,source_month,ingested_at,game_outcome,rating_difference


In [0]:
%sql
SELECT game_id,
       COUNT(game_id) as count
FROM workspace.gold.fct_player_games
GROUP by game_id
HAVING COUNT(game_id)>1

game_id,count


In [0]:
%sql
SELECT player_username
FROM workspace.gold.fct_player_games
WHERE player_username IS NULL

player_username


In [0]:
%sql
SELECT opponent_username
FROM workspace.gold.fct_player_games
WHERE opponent_username IS NULL

opponent_username


In [0]:
%sql
SELECT game_outcome
FROM workspace.gold.fct_player_games
WHERE game_outcome NOT IN ('win','loss','draw')

game_outcome


In [0]:
%sql
SELECT player_color
FROM workspace.gold.fct_player_games
WHERE player_color NOT IN ('white','black')

player_color


In [0]:
%sql
SELECT
    (SELECT COUNT(*) FROM workspace.silver.games
     WHERE lower(white_username) = 'ajaza' OR lower(black_username) = 'ajaza') AS silver_ajaza_count,
    (SELECT COUNT(*) FROM workspace.gold.fct_player_games) AS gold_count

silver_ajaza_count,gold_count
10,10
